# Copying necessary steps from data_loading.py

In [163]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
import numpy as np
from importlib import reload
import data_loading  # Import the module instead of specific functions

reload(data_loading)  # Reload the module after making changes

# Now you can reference the functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

In [164]:
"""
Expands the categories in a given column of a DataFrame into separate binary columns.

Parameters:
- df: pandas.DataFrame, the DataFrame to modify.
- column_name: str, the name of the column to expand.
"""
def expand_categories_in_column(df, column_name):

    unique_categories = set()
    df[column_name].dropna().apply(lambda x: unique_categories.update(set(x.split(', '))))

    # init dict to hold new columns
    new_columns = {}
    
    for category in unique_categories:
        # Ensure category name is valid as a column name (e.g., no spaces or special characters)
        # add col name as there are duplicates across different cols
        valid_category_name = column_name + " / " + category.lower().replace(' ', '_').replace(',', '')
        
        # Instead of modifying df directly, create and store the new column in new_columns
        mask = df[column_name].fillna('').str.contains(category, regex=False, na=False)
        new_columns[valid_category_name] = mask.astype(int)

    # Create a new DataFrame from the new_columns dictionary
    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    
    # Concatenate the new columns to the original DataFrame
    df = pd.concat([df, new_columns_df], axis=1)

    

    return df

In [165]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
cad_string = current_script_directory / "../data/raw/canada/"
usa_string = current_script_directory  / "../data/raw/usa/"

cad_data = cad_string / "pipeline-incidents-comprehensive-data.csv"
cad_unknown_data = cad_string / "PODSdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv"

usa_pre1986 = usa_string / "accident_hazardous_liquid_pre1986/accident_hazardous_liquid_pre1986.txt"
usa_1986_jan2002 = usa_string / "accident_hazardous_liquid_1986_jan2002/accident_hazardous_liquid_1986_jan2002.txt"
usa_jan2002_dec2009 = usa_string / "accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt"
usa_jan2010_present = usa_string / "accident_hazardous_liquid_jan2010_present/accident_hazardous_liquid_jan2010_present.txt"
usa_gravity = usa_string / "accident_gravity_reporting_regulated_jul2020_present/accident_gravity_reporting_regulated_jul2020_present.txt"

# read data into dataframes
CAD_data_raw = read_csv_to_dataframe(cad_data)
CAD_unknown_data_raw = read_csv_to_dataframe(cad_unknown_data)

usa_pre1986_raw = read_txt_to_dataframe(usa_pre1986)
usa_1986_jan2002_raw = read_txt_to_dataframe(usa_1986_jan2002)
usa_jan2002_dec2009_raw = read_txt_to_dataframe(usa_jan2002_dec2009)
usa_jan2010_present_raw = read_txt_to_dataframe(usa_jan2010_present)
usa_gravity_raw = read_txt_to_dataframe(usa_gravity)

File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/pipeline-incidents-comprehensive-data.csv' successfully read into a DataFrame.
File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/PODSdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_pre1986/accident_hazardous_liquid_pre1986.txt' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_1986_jan2002/accident_hazardous_liquid_1986_jan2002.txt' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_l

# Cleaning

## CAD_data_raw

### All subtances

In [166]:
# make deep copy for cleaned df
CAD_data_cleaned = CAD_data_raw.copy()

# Generate profiling report to identify outliers/trends etc. for cleansing purposes
# CAD_profile = ProfileReport(CAD_data_raw, title="CAD data raw Profiling Report", explorative=True)
# CAD_profile.to_file("CAD_report.html")

# report generated that there are no duplicate rows and that country column has constant value but it's going to be dropped below anyway

In [167]:
# looked at data dictionary for columns that could be relevant to substance/pipeline specifications/cause of incident
# want to analyze this subset for trends/correlations
## data dict and data don't line up so needed to manually change some; labelled with "## different"
subset_columns = [
"pipeline or facility type",
"pipeline or facility equipment involved",
"rupture",
"incident types", ## different
"conditions that resulted in the operation beyond limits",
"pipeline outside diameter (nps)",
"pipeline length (km)",
"substance carried",
"facility type", ## different
"facility latitude",
"facility longitude",
"longitude",
"latitude",
"nominal pipe size",
"material",
"material grade",
"schedule",
"design wall thickness (mm)",
"custom design wall thickness (mm)",
"actual wall thickness (mm)",
"licensed maximum operating pressure (kpa)",
"actual operating pressure at time of failure (kpa)",
"year of manufacture",
"most recent cathodic protection reading at incident site (mv vs. cu/cuso4)",
"weld type",
"seam type",
"coating location",
"coating type",
"coating condition",
"application method",
"year when the coating was applied",
"insulation installed",
"detailed what happened", ## diff
"what happened category", ## diff
"detailed why it happened", ## diff
"why it happened category" ## diff
]

# Taking the subset
CAD_data_cleaned = CAD_data_cleaned[subset_columns]

# drop columns that have more than x percentage of na's
# can play around with this value to see how it affects results
threshold = 0.5
CAD_data_cleaned = CAD_data_cleaned.loc[:, CAD_data_cleaned.isnull().mean() < 0.5]

In [168]:
## expand categorical cols with multiple options into unique binary cols
## e.g., incident types: "fire", "fire, release of substance"
# if there are 5 of these options, it means 15 possible combinations for 1-hot encoding, instead of just 5 for binary
columns_to_expand = [
    'what happened category',
    'detailed what happened',
    'detailed why it happened',
    'why it happened category',
    'incident types',
    'substance carried'
]

for column in columns_to_expand:
    CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, column)

CAD_data_cleaned.drop(columns=columns_to_expand, inplace=True)

In [169]:
# only going to take first of pipeline outside diameter values
CAD_data_cleaned['pipeline outside diameter (nps)'] = CAD_data_cleaned['pipeline outside diameter (nps)'].fillna(0).str.split(',').str[0].astype(float)

# some cols are binary so going to map those instead of 1-hot encoding
binary_cols = ['pipeline or facility equipment involved', 'rupture', 'insulation installed']
for col in binary_cols:
    CAD_data_cleaned[col] = CAD_data_cleaned[col].map({'Yes': 1, 'No': 0, np.nan: -1})

# need to 1-hot encode 'pipeline or facility type'
categorical_cols = CAD_data_cleaned.select_dtypes(include=['object']).columns
cad_encoded = pd.get_dummies(CAD_data_cleaned, columns=categorical_cols, drop_first=True)

In [175]:
corr_matrix = cad_encoded.corr()
high_corr_pairs = corr_matrix.unstack().sort_values(kind="quicksort", ascending=False)
high_corr_pairs = high_corr_pairs[(abs(high_corr_pairs) > 0.8) & (high_corr_pairs != 1)]
print(high_corr_pairs)

monitoring_use_of_procedures_or_practices_or_rules_detailed why it happened  inadequate_maintenance_of_standards_detailed why it happened                   0.982543
inadequate_maintenance_of_standards_detailed why it happened                 monitoring_use_of_procedures_or_practices_or_rules_detailed why it happened    0.982543
geotechnical_failure_detailed what happened                                  natural_force_damage_what happened category                                    0.973930
natural_force_damage_what happened category                                  geotechnical_failure_detailed what happened                                    0.973930
unknown_detailed what happened                                               other_causes_what happened category                                            0.963268
                                                                                                                                                              ...   
poor_condi

In [ ]:
# want to look at incidents that could have been influenced by the material
# e.g., corrosion/cracking
# hard to determine... leaving this for now
# reasons_df = CAD_data_cleaned.copy()[['Detailed what happened', 'What happened category', 'Detailed why it happened', 'Why it happened category']].drop_duplicates()
# reasons_df['Why it happened category'].unique()

## columns that could be related to material influencing an accident/corrosion

#print(CAD_data_cleaned['Pipeline or Facility Type'].unique())
#CAD_data_cleaned['Rupture'].unique() ## loss of containment and unable to operate
#CAD_data_cleaned['Regulation'].unique()
#print(CAD_data_cleaned['Facility Type'].unique())

## Crude Only

In [ ]:
# only care about crude_oil incidents
CAD_data_raw = CAD_data_raw.loc[CAD_data_raw.Substance.str.contains("Crude Oil")]

# Crude Oil - Sour, Crude Oil - Sweet, Crude Oil - Synthetic
## LOOKING for whether the variations in sulfur (or synthetic?) affect incident rates
sorted(CAD_data_raw.Substance.unique())

#list(sorted(CAD_data_raw.columns))

Summarize dataset:  10%|█         | 11/107 [00:00<00:03, 30.82it/s, Describe variable:Incident Number]                /Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/ydata_profiling/model/pandas/summary_pandas.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  series = series.fillna(np.nan)
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/ydata_profiling/model/pandas/summary_pandas.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_sil